# Mapa Interactivo: AGEBs de Hermosillo

Visualización de las 665 AGEBs con tres capas sobrelapadas:
- **Valor catastral** ($/m²) — obtenido del catastro municipal 2025
- **Score de rezago habitacional** — promedio de 7 indicadores de carencia
- **Abandono alto** — variable objetivo (clase 1 = alto riesgo)

Cada AGEB muestra al hacer clic: valor catastral, score rezago, clase, negocios DENUE.

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import FloatImage
import json
from pathlib import Path

ROOT = Path('..')

# Cargar GeoJSON con todas las features por AGEB
gdf = gpd.read_file(ROOT / 'data/processed/maps/agebs_hermosillo_completo.geojson')
gdf = gdf.to_crs('EPSG:4326')

print(f'AGEBs: {len(gdf)}')
print(f'Columnas: {[c for c in gdf.columns if c != "geometry"]}')
print()
print(gdf.drop(columns='geometry').describe().round(2))

In [ ]:
# Centroide de Hermosillo
centro = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
print(f'Centro del mapa: {centro}')

# Preparar GeoJSON como dict para folium
geojson_data = json.loads(gdf.to_json())

## Mapa 1: Valor Catastral por AGEB ($/m²)

In [ ]:
m1 = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=geojson_data,
    data=gdf,
    columns=['CVE_AGEB', 'VALOR_CATASTRAL_MAX'],
    key_on='feature.properties.CVE_AGEB',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Valor Catastral Unitario ($/m²)',
    nan_fill_color='#d3d3d3',
    nan_fill_opacity=0.4,
    name='Valor Catastral'
).add_to(m1)

# Tooltips con información al pasar el mouse
tooltip_layer = folium.GeoJson(
    geojson_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['CVE_AGEB', 'VALOR_CATASTRAL_MAX', 'SCORE_REZAGO',
                'abandono_alto', 'n_bancos', 'n_cafes', 'n_inmobiliarias', 'n_total_neg'],
        aliases=['AGEB:', 'Valor Catastral ($/m²):', 'Score Rezago:',
                 'Abandono Alto:', 'Bancos:', 'Cafés:', 'Inmobiliarias:', 'Total Negocios:'],
        localize=True,
        sticky=False
    )
).add_to(m1)

folium.LayerControl().add_to(m1)
m1

## Mapa 2: Score de Rezago Habitacional

In [ ]:
m2 = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=geojson_data,
    data=gdf,
    columns=['CVE_AGEB', 'SCORE_REZAGO'],
    key_on='feature.properties.CVE_AGEB',
    fill_color='Blues',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Score de Rezago Habitacional (0–1)',
    name='Rezago Habitacional'
).add_to(m2)

folium.GeoJson(
    geojson_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['CVE_AGEB', 'SCORE_REZAGO', 'VALOR_CATASTRAL_MAX', 'abandono_alto'],
        aliases=['AGEB:', 'Score Rezago:', 'Valor Catastral ($/m²):', 'Abandono Alto:'],
        localize=True
    )
).add_to(m2)

folium.LayerControl().add_to(m2)
m2

## Mapa 3: Zonas de Alto Abandono (variable objetivo)

In [ ]:
m3 = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

# Color manual: rojo = abandono alto, verde = estable, gris = sin dato
def estilo_abandono(feature):
    val = feature['properties'].get('abandono_alto')
    if val is None:
        color = '#d3d3d3'
    elif val == 1:
        color = '#d73027'
    else:
        color = '#1a9850'
    return {'fillColor': color, 'color': '#555', 'weight': 0.5,
            'fillOpacity': 0.7}

folium.GeoJson(
    geojson_data,
    style_function=estilo_abandono,
    tooltip=folium.GeoJsonTooltip(
        fields=['CVE_AGEB', 'abandono_alto', 'VALOR_CATASTRAL_MAX',
                'SCORE_REZAGO', 'n_total_neg'],
        aliases=['AGEB:', 'Abandono Alto (1=sí):', 'Valor Catastral ($/m²):',
                 'Score Rezago:', 'Total Negocios DENUE:'],
        localize=True
    ),
    name='Abandono Alto'
).add_to(m3)

# Leyenda manual
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index:9999;
     background: white; padding: 10px; border-radius: 5px;
     border: 1px solid #aaa; font-size: 13px;">
  <b>Abandono de Vivienda</b><br>
  <span style="color:#1a9850">&#9632;</span> Estable (clase 0)<br>
  <span style="color:#d73027">&#9632;</span> Alto riesgo (clase 1)<br>
  <span style="color:#d3d3d3">&#9632;</span> Sin dato
</div>
'''
m3.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m3)
m3

## Mapa 4: Actividad Económica DENUE (total negocios por AGEB)

In [ ]:
m4 = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=geojson_data,
    data=gdf,
    columns=['CVE_AGEB', 'n_total_neg'],
    key_on='feature.properties.CVE_AGEB',
    fill_color='PuBuGn',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Total Negocios DENUE por AGEB',
    name='Actividad Económica'
).add_to(m4)

folium.GeoJson(
    geojson_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['CVE_AGEB', 'n_total_neg', 'n_bancos', 'n_cafes',
                'n_inmobiliarias', 'VALOR_CATASTRAL_MAX', 'abandono_alto'],
        aliases=['AGEB:', 'Total Negocios:', 'Bancos:', 'Cafés:',
                 'Inmobiliarias:', 'Valor Catastral ($/m²):', 'Abandono Alto:'],
        localize=True
    )
).add_to(m4)

folium.LayerControl().add_to(m4)
m4

## Mapa 5: El Punto Ciego — Valor Catastral vs Actividad Económica

Este mapa muestra exactamente el problema que motivó la feature catastral:
- **Verde oscuro** = alto valor catastral + muchos negocios (zona comercial rica)
- **Amarillo** = alto valor catastral + pocos negocios (**zona de lujo residencial — el punto ciego**)
- **Azul** = bajo valor + muchos negocios (zona comercial popular)
- **Rojo/gris** = bajo valor + pocos negocios (zona marginal o abandonada)

In [ ]:
import numpy as np

# Normalizar ambas variables a [0,1]
v_max = gdf['VALOR_CATASTRAL_MAX'].max()
n_max = gdf['n_total_neg'].max()

gdf['v_norm'] = gdf['VALOR_CATASTRAL_MAX'] / v_max
gdf['n_norm'] = gdf['n_total_neg'] / n_max

def color_bivar(v_norm, n_norm):
    """Bivariado: valor (eje Y) vs negocios (eje X)"""
    r = int(255 * (1 - v_norm))      # menos valor → más rojo
    g = int(200 * v_norm * n_norm)   # ambos altos → verde
    b = int(255 * (1 - n_norm))      # menos negocios → más azul
    return f'rgb({r},{g},{b})'

m5 = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

geojson_data5 = json.loads(gdf.to_json())

# Enriquecer con el color calculado
ageb_colors = {
    row['CVE_AGEB']: color_bivar(row['v_norm'], row['n_norm'])
    for _, row in gdf.iterrows()
}

def estilo_bivar(feature):
    ageb = feature['properties']['CVE_AGEB']
    color = ageb_colors.get(ageb, '#d3d3d3')
    return {'fillColor': color, 'color': '#444', 'weight': 0.4, 'fillOpacity': 0.8}

folium.GeoJson(
    geojson_data5,
    style_function=estilo_bivar,
    tooltip=folium.GeoJsonTooltip(
        fields=['CVE_AGEB', 'VALOR_CATASTRAL_MAX', 'n_total_neg', 'abandono_alto', 'SCORE_REZAGO'],
        aliases=['AGEB:', 'Valor Catastral ($/m²):', 'Negocios DENUE:', 'Abandono Alto:', 'Score Rezago:'],
        localize=True
    ),
    name='Valor vs Negocios'
).add_to(m5)

legend5 = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:9999;
     background:white; padding:12px; border-radius:6px;
     border:1px solid #aaa; font-size:12px; line-height:1.8;">
  <b>Valor Catastral vs Actividad DENUE</b><br>
  <span style="background:rgb(0,200,0);padding:0 8px;">&nbsp;</span> Alto valor + muchos negocios<br>
  <span style="background:rgb(0,0,255);padding:0 8px;">&nbsp;</span> Alto valor + pocos negocios <b>(lujo residencial)</b><br>
  <span style="background:rgb(255,100,0);padding:0 8px;">&nbsp;</span> Bajo valor + muchos negocios<br>
  <span style="background:rgb(255,0,255);padding:0 8px;">&nbsp;</span> Bajo valor + pocos negocios (marginal)<br>
</div>
'''
m5.get_root().html.add_child(folium.Element(legend5))
folium.LayerControl().add_to(m5)
m5

## Exportar mapas como HTML

Guardar para incluir en el reporte o presentación.

In [ ]:
OUT = ROOT / 'data/processed/maps'
m1.save(str(OUT / 'mapa_catastral.html'))
m2.save(str(OUT / 'mapa_rezago.html'))
m3.save(str(OUT / 'mapa_abandono.html'))
m4.save(str(OUT / 'mapa_denue.html'))
m5.save(str(OUT / 'mapa_bivar_valor_denue.html'))

print('Mapas exportados:')
for f in ['mapa_catastral.html','mapa_rezago.html','mapa_abandono.html',
          'mapa_denue.html','mapa_bivar_valor_denue.html']:
    print(f'  data/processed/maps/{f}')